# 02 Keyword Search

Import Libraries

In [1]:
import pandas as pd
import re
from pathlib import Path

Load cleaned data

In [7]:
cleaned_data_path = Path(
    "../../data/processed/keyword_ready_messages.csv"
)


In [8]:
keyword_df = pd.read_csv(cleaned_data_path)

Prevent missing text from causing matching errors

In [9]:
keyword_df["clean_text"] = (
    keyword_df["clean_text"]
    .fillna("")
    .astype(str)
)

In [10]:
print("Cleaned data load successed")
print('Available columns:', keyword_df.columns.tolist())

Cleaned data load successed
Available columns: ['userID', 'time', 'role', 'message', 'detected_ad_name', 'message_english', 'is_ad_entry', 'clean_text', 'is_low_information']


Keyword Categories

Keywords are grouped according to products, customer concerns, reasons for
contacting the chatbot, and campaign-related interest.

The initial dictionary is based on the project objectives and common terms in
loan and insurance conversations. It will be improved after reviewing matched
and unmatched messages.

In [11]:
keyword_categories = {
    # Product interest
    "loan_product": [
        "loan",
        "car loan",
        "motorcycle loan",
        "land loan",
        "refinance",
        "financing",
        "credit"
    ],

    "insurance_product": [
        "insurance",
        "car insurance",
        "motorcycle insurance",
        "policy",
        "coverage"
    ],

    # Loan and insurance concerns
    "eligibility": [
        "eligible",
        "eligibility",
        "qualify",
        "qualification",
        "salary",
        "income",
        "age requirement"
    ],

    "application_process": [
        "apply",
        "application",
        "register",
        "registration",
        "submit"
    ],

    "required_documents": [
        "document",
        "documents",
        "paperwork",
        "required document"
    ],

    "approval_and_status": [
        "approve",
        "approved",
        "approval",
        "application status",
        "check status",
        "pending",
        "rejected"
    ],

    "interest_and_fees": [
        "interest",
        "interest rate",
        "fee",
        "fees",
        "charge",
        "charges"
    ],

    "credit_limit": [
        "credit limit",
        "loan limit",
        "increase the credit limit",
        "loan amount",
        "credit amount"
    ],

    "payment_and_installment": [
        "payment",
        "pay",
        "installment",
        "monthly payment",
        "repayment",
        "payment channel",
        "overdue"
    ],

    "insurance_premium": [
        "premium",
        "insurance premium",
        "premium price",
        "insurance price"
    ],

    "insurance_claim": [
        "claim",
        "make a claim",
        "claim status",
        "accident",
        "damage"
    ],

    "coverage_and_conditions": [
        "coverage",
        "condition",
        "conditions",
        "requirement",
        "requirements",
        "benefit",
        "benefits",
        "protection"
    ],

    "renewal_and_cancellation": [
        "renew",
        "renewal",
        "expired",
        "expiration",
        "cancel",
        "cancellation"
    ],

    # Campaign questions and interest
    "campaign_or_promotion": [
        "campaign",
        "promotion",
        "promotional",
        "offer",
        "special offer",
        "discount",
        "privilege",
        "reward"
    ],

    "campaign_interest": [
        "join",
        "participate",
        "interested",
        "register",
        "receive the right",
        "receive rights",
        "get the offer"
    ],

    # Other common reasons for contact
    "branch_or_contact": [
        "branch",
        "location",
        "contact",
        "phone number",
        "call me",
        "contact me"
    ]
}

Create matching function
-- The function uses word boundaries to reduce patial-word flase matches.

In [12]:
def find_keyword_matches(text, keywords):
    matches = []

    for keyword in keywords:
        pattern = rf"(?<!\w){re.escape(keyword)}(?!\w)"

        if re.search(pattern, text, flags=re.IGNORECASE):
            matches.append(keyword)

    return matches

Text fun_: with artifical text

In [13]:
test_message = "what documents do i need to apply for a car loan"

for category, keywords in keyword_categories.items():
    matches = find_keyword_matches(test_message, keywords)

    if matches:
        print(f"{category}: {matches}")

loan_product: ['loan', 'car loan']
application_process: ['apply']
required_documents: ['documents']


Apply keyword matching

Match list column for each category

In [15]:
for category, keywords in keyword_categories.items():
    keyword_df[f"{category}_matches"] = keyword_df["clean_text"].apply(
        lambda text: find_keyword_matches(text, keywords)
    )

Boolean category columns

In [16]:
for category in keyword_categories:
    keyword_df[category] = keyword_df[
        f"{category}_matches"
    ].apply(bool)

Record all matched categories and keywords

In [18]:
def get_matched_categories(row):
    return[
        category
        for category in keyword_categories
        if row[category]
    ]

def get_all_matched_keywords(row):
    results = []

    for category in keyword_categories:
        for keyword in row[f"{category}_matches"]:
            results.append(f'{category}:{keyword}')

    return results

In [19]:
keyword_df['matched_categories'] = keyword_df.apply(
    get_matched_categories,
    axis=1
)

keyword_df['matched_keywords'] = keyword_df.apply(
    get_all_matched_keywords,
    axis=1
)

keyword_df["has_keyword_match"] = keyword_df[
    "matched_categories"
].apply(bool)

Check the overall matching result

In [20]:
matched_count = keyword_df["has_keyword_match"].sum()
unmatched_count = (~keyword_df["has_keyword_match"]).sum()
total_messages = len(keyword_df)

matched_percentage = matched_count / total_messages * 100
unmatched_percentage = unmatched_count / total_messages * 100

print(f"Matched messages: {matched_count:,} ({matched_percentage:.2f}%)")
print(f"Unmatched messages: {unmatched_count:,} ({unmatched_percentage:.2f}%)")

Matched messages: 2,709 (41.52%)
Unmatched messages: 3,816 (58.48%)


Category summary

In [21]:
category_summary = pd.DataFrame({
    "category": list(keyword_categories.keys()),
    "message_count": [
        keyword_df[category].sum()
        for category in keyword_categories
    ]
})

category_summary["percentage"] = (
    category_summary["message_count"]
    / len(keyword_df)
    * 100
).round(2)

category_summary = category_summary.sort_values(
    "message_count",
    ascending=False
).reset_index(drop=True)

category_summary

,category,message_count,percentage
0,loan_product,1687,25.85
1,coverage_and_conditions,258,3.95
2,branch_or_contact,232,3.56
3,insurance_product,207,3.17
4,payment_and_installment,198,3.03
5,application_process,180,2.76
6,campaign_interest,153,2.34
7,required_documents,146,2.24
8,credit_limit,75,1.15
9,eligibility,41,0.63


Validate matched messages

In [22]:
matched_sample = keyword_df.loc[
    keyword_df["has_keyword_match"],
    [
        "clean_text",
        "matched_categories",
        "matched_keywords"
    ]
].sample(
    n=min(20, matched_count),
    random_state=42
)

matched_sample

,clean_text,matched_categories,matched_keywords
6431,is there a refinance from anywhere else,[loan_product],[loan_product:refinance]
5695,i chose the dan khun thot branch the nearby lo...,[branch_or_contact],"[branch_or_contact:branch, branch_or_contact:l..."
1095,motorcycle loan or refinance,[loan_product],"[loan_product:loan, loan_product:motorcycle lo..."
678,loan details,[loan_product],[loan_product:loan]
3189,conditions for applying for a motorcycle loan,"[loan_product, coverage_and_conditions]","[loan_product:loan, loan_product:motorcycle lo..."
2412,condition,[coverage_and_conditions],[coverage_and_conditions:condition]
5237,motorcycle loan or refinance,[loan_product],"[loan_product:loan, loan_product:motorcycle lo..."
975,request a credit limit increase check eligibil...,"[loan_product, eligibility, credit_limit]","[loan_product:loan, loan_product:credit, eligi..."
4811,if the loan doesn't go through do i have to ge...,[loan_product],[loan_product:loan]
723,not yet completed in installments conditions f...,[coverage_and_conditions],[coverage_and_conditions:conditions]


Review unmatched messages

In [23]:
unmatched_df = keyword_df.loc[
    ~keyword_df["has_keyword_match"]
].copy()

unmatched_sample = unmatched_df[
    ["clean_text"]
].sample(
    n=min(20, len(unmatched_df)),
    random_state=42
)

unmatched_sample

,clean_text
1969,bt-50
3411,pawn the book
4660,motorcycle
3471,2500
2171,i just made a mistake with toyota
2011,isuzu
4587,3016
3002,can i
1302,year 2013
1725,year 1995


Save the keyword results locally

In [24]:
keyword_results = keyword_df.copy()

list_columns = [
    "matched_categories",
    "matched_keywords"
] + [
    f"{category}_matches"
    for category in keyword_categories
]

for column in list_columns:
    keyword_results[column] = keyword_results[column].apply(
        lambda values: " | ".join(values)
    )

In [25]:
output_path = Path(
    "../../data/processed/keyword_search_results.csv"
)

keyword_results.to_csv(output_path, index=False)

print("Keyword results saved successfully.")

Keyword results saved successfully.
